# Defensive Shot Suppression Analysis

This notebook analyzes whether teams that allowed opponents into their final third also allowed those opponents to convert that access into shots during the 2026 FIFA World Cup.

The analysis uses team-level FIFA Match Centre data and focuses on the relationship between opponent final-third entries and opponent shots.

## Main Question

When a team allowed the opponent into its final third, how often did that access turn into an actual shot against them?

## Key Metric

Entry-to-Shot Conceded Rate = opponent attempts at goal / opponent final-third entries × 100

A lower value is better.

It means the team allowed the opponent to enter its defensive territory, but limited the opponent's ability to turn that access into shots.

## Why This Matters

This flips the earlier attacking analyses in the series.

Instead of asking:

"Did our territory access turn into shots?"

this notebook asks:

"Did the opponent's territory access turn into shots against us?"

This is different from simply allowing fewer entries. A team can allow territory but still defend the shot-creation phase well.

## Data Source

FIFA Match Centre, full FIFA Official Stats only.

Belgium vs Egypt is excluded from full-stat analysis because FIFA provides only Live Statistics for that match.

## Important Notes

- Rankings are descriptive, not causal.
- Match counts differ from 3 to 8, so smaller samples may be more volatile.
- xG, shot location, defensive pressure, and opponent strength are not controlled.
- This is a shot-suppression proxy, not a complete defensive quality metric.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional label adjustment library
try:
    from adjustText import adjust_text
    ADJUST_TEXT_AVAILABLE = True
except ImportError:
    ADJUST_TEXT_AVAILABLE = False

# =========================================================
# Edit only this line if the project folder is moved
# =========================================================
BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break

DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Input file:", RAW_CSV_PATH)
print("Output directory:", OUTPUT_DIR)
print("adjustText available:", ADJUST_TEXT_AVAILABLE)

In [ ]:
# =========================================================
# Step 2. Load the FIFA official stats dataset
# =========================================================

raw = pd.read_csv(RAW_CSV_PATH)

print("Raw dataset shape:", raw.shape)
print("Number of match-team rows:", len(raw))
print("Number of matches:", raw["match_id"].nunique())
print("Number of teams:", raw["team_name"].nunique())

display(raw.head())

In [ ]:
# =========================================================
# Step 2.5. Check incomplete matches and missing values
# =========================================================

# Columns needed for this analysis
defensive_analysis_columns = [
    "match_id",
    "team_name",
    "opponent_name",
    "stats_complete",
    "stats_source",
    "attacking__attempts_at_goal__total",
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

# Check whether required columns exist before inspecting missing values.
missing_columns = [col for col in defensive_analysis_columns if col not in raw.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All defensive analysis columns are available.")

# Check rows that are not full FIFA Official Stats.
incomplete_rows = raw[raw["stats_complete"] != True].copy()

print("Rows without full FIFA Official Stats:", len(incomplete_rows))
print("Matches without full FIFA Official Stats:", incomplete_rows["match_id"].nunique())

if len(incomplete_rows) > 0:
    display(
        incomplete_rows[
            [
                "match_id",
                "team_name",
                "opponent_name",
                "stats_complete",
                "stats_source",
                "attacking__attempts_at_goal__total",
                "attacking__final_third_entries__left_channel",
                "attacking__final_third_entries__left_inside_channel",
                "attacking__final_third_entries__central_channel",
                "attacking__final_third_entries__right_inside_channel",
                "attacking__final_third_entries__right_channel",
            ]
        ]
    )

# Specific check for Belgium vs Egypt.
belgium_egypt_check = raw[
    raw["team_name"].isin(["Belgium", "Egypt"])
    & raw["opponent_name"].isin(["Belgium", "Egypt"])
].copy()

print("Belgium vs Egypt rows found:", len(belgium_egypt_check))

if len(belgium_egypt_check) > 0:
    display(
        belgium_egypt_check[
            [
                "match_id",
                "team_name",
                "opponent_name",
                "stats_complete",
                "stats_source",
                "attacking__attempts_at_goal__total",
                "attacking__final_third_entries__left_channel",
                "attacking__final_third_entries__left_inside_channel",
                "attacking__final_third_entries__central_channel",
                "attacking__final_third_entries__right_inside_channel",
                "attacking__final_third_entries__right_channel",
            ]
        ]
    )

# Missing value summary for all rows.
missing_summary_all = (
    raw[defensive_analysis_columns]
    .isna()
    .sum()
    .reset_index()
)

missing_summary_all.columns = ["column", "missing_values"]

display(missing_summary_all)


In [ ]:
# =========================================================
# Step 3. Keep only completed matches with full FIFA Official Stats
# =========================================================

full_stats = raw[raw["stats_complete"] == True].copy()

print("Rows before filtering:", len(raw))
print("Rows after keeping full FIFA Official Stats only:", len(full_stats))
print("Rows removed:", len(raw) - len(full_stats))

print("Matches before filtering:", raw["match_id"].nunique())
print("Matches after filtering:", full_stats["match_id"].nunique())
print("Matches removed:", raw["match_id"].nunique() - full_stats["match_id"].nunique())

removed_matches = (
    raw[raw["stats_complete"] != True]
    [
        [
            "match_id",
            "team_name",
            "opponent_name",
            "stats_complete",
            "stats_source",
        ]
    ]
    .copy()
)

if len(removed_matches) > 0:
    print("Removed match-team rows:")
    display(removed_matches)

In [ ]:
 # =========================================================
# Step 4. Create team attacking metrics used for opponent-side defensive analysis
# =========================================================

analysis_df = full_stats.copy()

# FIFA final-third entries are split into five attacking channels.
final_third_entry_columns = [
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

# Convert required numeric columns safely.
numeric_columns = final_third_entry_columns + [
    "attacking__attempts_at_goal__total",
]

for col in numeric_columns:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

# Total final-third entries made by each team in each match.
# min_count=5 ensures a row with all five channels missing stays NaN
# instead of silently becoming 0 (house rule for multi-column sums).
analysis_df["team_final_third_entries"] = analysis_df[final_third_entry_columns].sum(axis=1, min_count=5)

# Total attempts at goal made by each team in each match.
analysis_df["team_attempts_at_goal"] = analysis_df["attacking__attempts_at_goal__total"]

# Preview the metrics that will later become opponent metrics.
preview_columns = [
    "match_id",
    "team_name",
    "opponent_name",
    "team_final_third_entries",
    "team_attempts_at_goal",
]

display(analysis_df[preview_columns].head(10).round(2))

In [ ]:
# =========================================================
# Step 5. Flip attacking metrics into defensive conceded metrics
# =========================================================

# Keep the opponent-side attacking metrics that each team allowed.
opponent_metrics = analysis_df[
    [
        "match_id",
        "team_name",
        "team_final_third_entries",
        "team_attempts_at_goal",
    ]
].copy()

# Rename opponent_metrics columns so they can be joined back to the defending team.
opponent_metrics = opponent_metrics.rename(
    columns={
        "team_name": "opponent_name",
        "team_final_third_entries": "opponent_final_third_entries_conceded",
        "team_attempts_at_goal": "shots_conceded",
    }
)

# Join each team row with its opponent's attacking metrics from the same match.
defensive_match_df = analysis_df.merge(
    opponent_metrics,
    on=["match_id", "opponent_name"],
    how="left",
)

# Calculate how often opponent final-third entries became shots.
# Lower is better from the defending team's perspective.
defensive_match_df["entry_to_shot_conceded_rate"] = np.where(
    defensive_match_df["opponent_final_third_entries_conceded"] > 0,
    defensive_match_df["shots_conceded"]
    / defensive_match_df["opponent_final_third_entries_conceded"]
    * 100,
    np.nan,
)

# Preview defensive match-level metrics.
preview_columns = [
    "match_id",
    "team_name",
    "opponent_name",
    "opponent_final_third_entries_conceded",
    "shots_conceded",
    "entry_to_shot_conceded_rate",
]

display(defensive_match_df[preview_columns].head(10).round(2))

In [ ]:
# =========================================================
# Step 6. Aggregate defensive shot suppression metrics by team
# =========================================================

team_defensive_metrics = (
    defensive_match_df
    .groupby("team_name", as_index=False)
    .agg(
        matches_played=("match_id", "nunique"),
        opponent_final_third_entries_conceded=(
            "opponent_final_third_entries_conceded",
            "sum",
        ),
        shots_conceded=("shots_conceded", "sum"),
    )
)

# Calculate team-level Entry-to-Shot Conceded Rate.
# Lower is better: fewer opponent entries turned into shots.
team_defensive_metrics["entry_to_shot_conceded_rate"] = np.where(
    team_defensive_metrics["opponent_final_third_entries_conceded"] > 0,
    team_defensive_metrics["shots_conceded"]
    / team_defensive_metrics["opponent_final_third_entries_conceded"]
    * 100,
    np.nan,
)

# Per-match volume metrics for easier interpretation.
team_defensive_metrics["opponent_final_third_entries_conceded_per_match"] = (
    team_defensive_metrics["opponent_final_third_entries_conceded"]
    / team_defensive_metrics["matches_played"]
)

team_defensive_metrics["shots_conceded_per_match"] = (
    team_defensive_metrics["shots_conceded"]
    / team_defensive_metrics["matches_played"]
)

# Display a clean preview.
display(
    team_defensive_metrics
    .sort_values("entry_to_shot_conceded_rate", ascending=True)
    .round(2)
    .head(10)
)

In [ ]:
# =========================================================
# Step 7. Review best and worst defensive shot suppression teams
# =========================================================

semifinalists = ["Spain", "France", "Argentina", "England"]

# Lower Entry-to-Shot Conceded Rate is better.
best_suppression_teams = (
    team_defensive_metrics
    .sort_values("entry_to_shot_conceded_rate", ascending=True)
    .copy()
)

worst_suppression_teams = (
    team_defensive_metrics
    .sort_values("entry_to_shot_conceded_rate", ascending=False)
    .copy()
)

semifinalist_defensive_metrics = (
    team_defensive_metrics[
        team_defensive_metrics["team_name"].isin(semifinalists)
    ]
    .copy()
    .sort_values("entry_to_shot_conceded_rate", ascending=True)
)

print("Best defensive shot suppression teams: lower rate is better")
display(
    best_suppression_teams[
        [
            "team_name",
            "matches_played",
            "opponent_final_third_entries_conceded",
            "shots_conceded",
            "entry_to_shot_conceded_rate",
            "opponent_final_third_entries_conceded_per_match",
            "shots_conceded_per_match",
        ]
    ]
    .round(2)
    .head(10)
)

print("Worst defensive shot suppression teams: higher rate is worse")
display(
    worst_suppression_teams[
        [
            "team_name",
            "matches_played",
            "opponent_final_third_entries_conceded",
            "shots_conceded",
            "entry_to_shot_conceded_rate",
            "opponent_final_third_entries_conceded_per_match",
            "shots_conceded_per_match",
        ]
    ]
    .round(2)
    .head(10)
)

print("Semifinalists only")
display(
    semifinalist_defensive_metrics[
        [
            "team_name",
            "matches_played",
            "opponent_final_third_entries_conceded",
            "shots_conceded",
            "entry_to_shot_conceded_rate",
            "opponent_final_third_entries_conceded_per_match",
            "shots_conceded_per_match",
        ]
    ]
    .round(2)
)

In [ ]:
# =========================================================
# Step 7.5. Verify semifinalist gap and check robustness
# =========================================================
# This cell calculates the actual gap between semifinalists and the
# rest of the field, then re-checks it using only teams with 4+ matches.
# Smaller-sample teams (3 matches) can swing to extreme values, so this
# second number tells us how much of the gap survives once we remove
# that noise -- same check used in notebook 07.

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

sf = team_defensive_metrics[team_defensive_metrics.team_name.isin(SEMIFINALISTS)]
rest = team_defensive_metrics[~team_defensive_metrics.team_name.isin(SEMIFINALISTS)]

sf_avg = sf["entry_to_shot_conceded_rate"].mean()
rest_avg = rest["entry_to_shot_conceded_rate"].mean()
gap = (sf_avg - rest_avg) / rest_avg * 100

print(f"All 48 teams — Semifinalists: {sf_avg:.2f}%, Rest: {rest_avg:.2f}%, Gap: {gap:.1f}%")

# Robustness check: teams with 4+ matches only.
robust = team_defensive_metrics[team_defensive_metrics["matches_played"] >= 4]
sf_r = robust[robust.team_name.isin(SEMIFINALISTS)]
rest_r = robust[~robust.team_name.isin(SEMIFINALISTS)]

sf_r_avg = sf_r["entry_to_shot_conceded_rate"].mean()
rest_r_avg = rest_r["entry_to_shot_conceded_rate"].mean()
gap_r = (sf_r_avg - rest_r_avg) / rest_r_avg * 100

print(f"4+ matches only ({len(robust)} teams) — Semifinalists: {sf_r_avg:.2f}%, Rest: {rest_r_avg:.2f}%, Gap: {gap_r:.1f}%")

In [ ]:
 # =========================================================
# Step 8. Scatter plot: Defensive shot suppression
# Final version: all team labels + selective callout lines
# =========================================================

x_col = "opponent_final_third_entries_conceded_per_match"
y_col = "entry_to_shot_conceded_rate"

plot_df = team_defensive_metrics.dropna(
    subset=[x_col, y_col]
).copy()

semifinalists = ["Spain", "France", "Argentina", "England"]

match_color_map = {
    3: "#8b5cf6",
    4: "#3b82f6",
    5: "#10b981",
    6: "#f59e0b",
    8: "#dc2626",
}

size_map = {
    3: 50,
    4: 68,
    5: 90,
    6: 115,
    8: 145,
}

plot_df["point_size"] = plot_df["matches_played"].map(size_map).fillna(70)

fig, ax = plt.subplots(figsize=(22, 11.5), dpi=150)

# Plot all teams as points.
for matches_played, group in plot_df.groupby("matches_played"):
    ax.scatter(
        group[x_col],
        group[y_col],
        s=group["point_size"],
        color=match_color_map.get(matches_played, "#64748b"),
        alpha=0.78,
        edgecolor="#334155",
        linewidth=0.55,
        label=f"{int(matches_played)} matches",
        zorder=3,
    )

# Median reference lines.
x_median = plot_df[x_col].median()
y_median = plot_df[y_col].median()

ax.axvline(
    x_median,
    linestyle="--",
    color="#94a3b8",
    linewidth=1.0,
    alpha=0.85,
)

ax.axhline(
    y_median,
    linestyle="--",
    color="#94a3b8",
    linewidth=1.0,
    alpha=0.85,
)

ax.text(
    x_median + 0.45,
    plot_df[y_col].max() - 1.0,
    "Median opponent entries conceded",
    fontsize=9.2,
    color="#64748b",
    va="top",
)

ax.text(
    plot_df[x_col].min() + 0.2,
    y_median + 0.25,
    "Median conceded rate",
    fontsize=9.2,
    color="#64748b",
    ha="left",
)

# Manual label offsets.
# These values move labels only, not the actual data points.
label_offsets = {
    # Semifinalists
    "Spain": (0.65, -0.12),
    "France": (0.65, 0.05),
    "Argentina": (0.75, -0.35),
    "England": (0.45, 0.10),

    # Selective callouts for the crowded cluster
    "Egypt": (-3.20, 1.35),
    "Norway": (2.40, 1.10),
    "Austria": (2.50, -0.85),

    # Small manual cleanup for nearby central teams
    "Senegal": (0.45, 0.45),
    "Sweden": (0.45, -0.35),
    "Netherlands": (-0.90, -0.25),
    "Côte d'Ivoire": (0.45, -0.28),
    "Morocco": (0.45, -0.40),
    "Panama": (0.45, -0.25),
    "Japan": (0.45, -0.35),
    "Bosnia and Herzegovina": (0.45, -0.20),
    "Congo DR": (0.45, 0.20),

    # Far-right teams
    "Paraguay": (0.45, -0.10),
    "Cabo Verde": (0.45, 0.12),
}

callout_teams = {"Egypt", "Norway", "Austria"}

for _, row in plot_df.iterrows():
    team = row["team_name"]
    x = row[x_col]
    y = row[y_col]

    dx, dy = label_offsets.get(team, (0.35, 0.15))
    is_semifinalist = team in semifinalists

    label_x = x + dx
    label_y = y + dy
    ha = "right" if dx < 0 else "left"

    # Use connector lines only for the three crowded callout teams.
    if team in callout_teams:
        ax.plot(
            [x, label_x],
            [y, label_y],
            color="#94a3b8",
            linewidth=0.8,
            alpha=0.75,
            zorder=2,
        )

    bbox_style = None
    if team in callout_teams:
        bbox_style = dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.80,
            pad=1.0,
        )

    ax.text(
        label_x,
        label_y,
        team,
        fontsize=9.0 if is_semifinalist else 7.7,
        fontweight="bold" if is_semifinalist else "normal",
        color="#111827" if is_semifinalist else "#475569",
        ha=ha,
        va="center",
        bbox=bbox_style,
        zorder=6 if is_semifinalist else 4,
    )

ax.set_title(
    "Defensive Shot Suppression: Territory Allowed vs Shots Conceded",
    fontsize=18,
    pad=18,
)

ax.set_xlabel(
    "Opponent Final-third Entries Conceded per Match",
    fontsize=12.5,
)

ax.set_ylabel(
    "Entry-to-Shot Conceded Rate (%)",
    fontsize=12.5,
)

ax.grid(alpha=0.20)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.legend(
    title="Matches played",
    loc="upper right",
    frameon=True,
    fontsize=10,
    title_fontsize=11,
    borderpad=0.7,
    labelspacing=0.6,
    handletextpad=0.7,
    markerscale=1.15,
)

footnote = (
    "Semifinalists conceded shots at a lower rate than the rest of the field "
    "(19.96% vs 23.62%, a 15.5% gap across all 48 teams; 10.0% among teams with 4+ matches). "
    "Lower Entry-to-Shot Conceded Rate is better. All teams are labelled. "
    "Belgium vs Egypt excluded because FIFA provides only Live Statistics for that match.\n"
    "Data source: FIFA Match Centre | Full FIFA Official Stats only | xG, shot location, and opponent strength not controlled."
)
fig.text(
    0.08,
    0.018,
    footnote,
    fontsize=8.5,
    color="gray",
)

plt.tight_layout(rect=[0, 0.055, 1, 1])

output_path = OUTPUT_DIR / "23_defensive_shot_suppression_scatter_all_labels_final.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")

plt.show()

print(f"Saved figure: {output_path}")